In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

# 1. Define the path to the test file (copied from your cell output)
test_file_path = '/kaggle/input/competitions/re-id-x-scalable-re-id-for-dynamic-camera-systems/test.csv'

# 2. Load the test data
test_df = pd.read_csv(test_file_path)

# 3. Create a 'Dummy' submission 
# The competition requires: row_id, frame_id, pred_track_id, xmin, ymin, xmax, ymax
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'frame_id': test_df['frame_id'],
    'pred_track_id': 0,        # Placeholder ID for every person found
    'xmin': 100, 'ymin': 100,  # Placeholder box coordinates
    'xmax': 200, 'ymax': 200
})

# 4. Save to the /kaggle/working directory
submission.to_csv('submission.csv', index=False)

print("Success! Created submission.csv with", len(submission), "rows.")

In [ ]:
# This requires internet to be ON in your sidebar settings
!pip install ultralytics -q

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# Load a pre-trained "Nano" model (small, fast, and great for baselines)
model = YOLO('yolov8n.pt')

In [ ]:
# Pick one image from your test folder
img_path = '/kaggle/input/competitions/re-id-x-scalable-re-id-for-dynamic-camera-systems/Dataset/test/images/test_0_000000.jpg'

results = model(img_path)

# This will draw boxes around the people it finds
for r in results:
    im_array = r.plot()
    plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
# 1. Use the 'track' method instead of just calling the model
# persist=True tells the model to remember IDs from the previous frame
img_folder = '/kaggle/input/competitions/re-id-x-scalable-re-id-for-dynamic-camera-systems/Dataset/test/images/'
test_images = sorted(os.listdir(img_folder))[:10] # Let's test the first 10 frames

results_list = []

for img_name in test_images:
    full_path = os.path.join(img_folder, img_name)
    
    # tracker='botsort.yaml' is a great default for Re-ID
    results = model.track(full_path, persist=True, tracker="botsort.yaml")
    
    for r in results:
        boxes = r.boxes
        for i in range(len(boxes)):
            # Get the ID assigned by the tracker
            track_id = int(boxes.id[i]) if boxes.id is not None else -1
            coords = boxes.xyxy[i].cpu().numpy() # [xmin, ymin, xmax, ymax]
            
            results_list.append({
                'frame_id': img_name,
                'pred_track_id': track_id,
                'xmin': coords[0], 'ymin': coords[1],
                'xmax': coords[2], 'ymax': coords[3]
            })

# View what we found
temp_df = pd.DataFrame(results_list)
print(temp_df.head())

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

# 1. Load the official test file
test_meta = pd.read_csv('/kaggle/input/competitions/re-id-x-scalable-re-id-for-dynamic-camera-systems/test.csv')
img_dir = '/kaggle/input/competitions/re-id-x-scalable-re-id-for-dynamic-camera-systems/Dataset/test/images/'

# 2. Use the Medium Model (Significant upgrade over Nano)
model = YOLO('yolov8m.pt')

final_predictions = []
print(f"Starting Version 10: YOLOv8m @ 640px for {len(test_meta)} frames...")

for i in range(len(test_meta)):
    row = test_meta.iloc[i]
    frame_id = row['frame_id']
    row_id = row['row_id']
    img_path = os.path.join(img_dir, frame_id)
    
    # 3. Optimized Parameters
    # imgsz=640: The native resolution for YOLOv8m (Fast & Balanced)
    # augment=False: Speeds up processing by 2x compared to Version 9
    results = model.track(
        img_path, 
        persist=True, 
        conf=0.25,      
        iou=0.45,       
        imgsz=640,      
        augment=False,  
        classes=0,      
        tracker="botsort.yaml", 
        verbose=False
    )
    
    # 4. Extracting Detections
    if results[0].boxes.id is not None:
        # Take the most confident detection
        box = results[0].boxes[0] 
        track_id = int(box.id[0])
        coords = box.xyxy[0].cpu().numpy()
        
        final_predictions.append({
            'row_id': row_id,
            'frame_id': frame_id,
            'pred_track_id': track_id,
            'xmin': int(round(coords[0])),
            'ymin': int(round(coords[1])),
            'xmax': int(round(coords[2])),
            'ymax': int(round(coords[3]))
        })
    else:
        # 5. Missing Row Imputation (Required for submission)
        final_predictions.append({
            'row_id': row_id, 
            'frame_id': frame_id,
            'pred_track_id': -1, 
            'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0
        })

    # Progress log every 10,000 frames
    if i % 10000 == 0:
        print(f"Progress: {i}/{len(test_meta)} frames ({(i/len(test_meta))*100:.1f}%)")

# 6. Final Submission Export
submission_df = pd.DataFrame(final_predictions)
submission_df.to_csv('submission.csv', index=False)
print(f"Success! Final submission file created with {len(submission_df)} rows.")